In [19]:
import boto3
import os
from flask import Flask, jsonify, request
from flask_cors import CORS
from datetime import datetime, timedelta
import time
from data_generation import generate_cell_tower_data, generate_predicted_outages
import uuid
from utils import n_towers_within_zone
from dotenv import load_dotenv
from decimal import Decimal


In [20]:
load_dotenv()

# --- AWS DynamoDB setup ---
dynamodb = boto3.resource(
    "dynamodb",
    region_name="us-east-1", # change region if needed
    aws_access_key_id=os.getenv("AWS_ACCESS_KEY"), # load from env variables
    aws_secret_access_key=os.getenv("AWS_SECRET_ACCESS_KEY")
)

# Change this to your DynamoDB table name
cell_towers_table = dynamodb.Table("cell-towers")
outages_table = dynamodb.Table("outages")

bedrock_agent_runtime_client = boto3.client('bedrock-agent-runtime', region_name='us-east-1')

In [21]:
# clear table
cell_towers_table.scan()['Items']
for tower_id in cell_towers_table.scan()['Items']:
    response = cell_towers_table.delete_item(Key={'tower_id': tower_id['tower_id']})

len(cell_towers_table.scan()['Items'])

0

In [22]:
cell_tower_data = generate_cell_tower_data(num_towers=1000)

for idx, tower_id in enumerate(cell_tower_data['tower_id']):
    item = {
        'tower_id': tower_id,
        'status': cell_tower_data['status'][idx],
        'longitude': Decimal(str(cell_tower_data['longitude'][idx])),
        'latitude': Decimal(str(cell_tower_data['latitude'][idx])),
        'signal_strength': Decimal(str(cell_tower_data['signal_strength'][idx])),
        'coverage_radius': Decimal(str(cell_tower_data['coverage_radius'][idx])),
        'bandwidth': cell_tower_data['bandwidth'][idx],
        'technology': cell_tower_data['technology'][idx],
    }
    response = cell_towers_table.put_item(Item=item)


In [23]:
len(cell_towers_table.scan()['Items'])

1000

In [24]:
cell_towers_table.scan()['Items'][:3]

[{'technology': '5G',
  'coverage_radius': Decimal('3.8171828602273896'),
  'signal_strength': Decimal('-102.89480370741786'),
  'tower_id': 'FN-1183',
  'status': 'Down',
  'longitude': Decimal('-104.20821225471231'),
  'latitude': Decimal('28.65929051830793'),
  'bandwidth': Decimal('100')},
 {'technology': '5G',
  'coverage_radius': Decimal('2.362099144478664'),
  'signal_strength': Decimal('-104.56665145978658'),
  'tower_id': 'FN-1021',
  'status': 'Down',
  'longitude': Decimal('-96.75495795400272'),
  'latitude': Decimal('29.045259681298685'),
  'bandwidth': Decimal('40')},
 {'technology': '4G LTE',
  'coverage_radius': Decimal('2.9429342964289154'),
  'signal_strength': Decimal('-66.46333133410681'),
  'tower_id': 'FN-1699',
  'status': 'Active',
  'longitude': Decimal('-97.11612768096302'),
  'latitude': Decimal('28.493832915148257'),
  'bandwidth': Decimal('40')}]

In [25]:
outage_data = generate_predicted_outages(20)
cell_tower_data = cell_towers_table.scan()['Items']

for idx, outage_id in enumerate(outage_data['outage_id']):
    item = {
        'outage_id': outage_id,
        'event': outage_data['event'][idx],
        'severity': outage_data['severity'][idx],
        'center_longitude': Decimal(str(outage_data['center_longitude'][idx])),
        'center_latitude': Decimal(str(outage_data['center_latitude'][idx])),
        'radius': Decimal(str(outage_data['radius'][idx]))
    }
    
    # get the number of towers within the zone
    n_towers, n_towers_down = n_towers_within_zone(outage_data['center_latitude'][idx], outage_data['center_longitude'][idx], outage_data['radius'][idx], cell_tower_data)
    
    item['towers_total'] = n_towers 
    item['towers_affected'] = n_towers_down
    
    response = outages_table.put_item(Item=item)


In [26]:
len(outages_table.scan()['Items'])

20

In [27]:
outages_table.scan()['Items'][:5]

[{'center_latitude': Decimal('35.256223516367555'),
  'event': 'Wildfire',
  'towers_affected': Decimal('3'),
  'towers_total': Decimal('7'),
  'radius': Decimal('41.214396087356086'),
  'center_longitude': Decimal('-103.96969679792443'),
  'outage_id': 'Outage-5',
  'severity': 'Low'},
 {'center_latitude': Decimal('33.36064305276954'),
  'event': 'Flood',
  'towers_affected': Decimal('0'),
  'towers_total': Decimal('1'),
  'radius': Decimal('17.974764676860985'),
  'center_longitude': Decimal('-105.93055104476602'),
  'outage_id': 'Outage-7',
  'severity': 'Low'},
 {'center_latitude': Decimal('29.888784397520524'),
  'event': 'Flood',
  'towers_affected': Decimal('0'),
  'towers_total': Decimal('1'),
  'radius': Decimal('42.470518822602685'),
  'center_longitude': Decimal('-101.89116241431171'),
  'outage_id': 'Outage-1',
  'severity': 'Low'},
 {'center_latitude': Decimal('26.441773478876495'),
  'event': 'Wildfire',
  'towers_affected': Decimal('1'),
  'towers_total': Decimal('3'),
 